# Safety Jacket Detection - Auto-Annotation & Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anzonathan/Traffic-Police-Agent/blob/main/notebooks/train_jacket_model_colab.ipynb)

> ⚠️ **Before running:** Set runtime to GPU — `Runtime → Change runtime type → T4 GPU`

**Dataset:** `UCUResearchLab/Reflector-Jackets` — 301 unannotated images.

**Pipeline:**
1. Install dependencies
2. Download dataset from HuggingFace
3. Save images with 80/20 train/val split
4. Auto-annotate with **Grounding DINO**
5. Train **YOLOv8**
6. Download `best.pt` to your local machine

## Step 1: Verify GPU & Install Dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q datasets transformers ultralytics pillow pyyaml accelerate
print("Dependencies installed!")

## Step 2: Download Dataset from HuggingFace

In [ ]:
import os, random
from datasets import load_dataset
from PIL import Image

DATASET_NAME = "UCUResearchLab/Reflector-Jackets"
print(f"Loading '{DATASET_NAME}'...")

ds = load_dataset(DATASET_NAME)
print(f"Dataset: {ds}")

# Auto-detect the image column
sample = ds['train'][0]
image_col = next(
    (k for k, v in sample.items() if hasattr(v, 'mode') and hasattr(v, 'save')),
    None
)
assert image_col, "No image column found. Inspect ds['train'].features above."
print(f"Image column: '{image_col}' | Total: {len(ds['train'])} images")

## Step 3: Save Images with 80/20 Train/Val Split

In [ ]:
for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

all_idx = list(range(len(ds['train'])))
random.seed(42)
random.shuffle(all_idx)

train_set = set(all_idx[:int(len(all_idx) * 0.8)])

for idx, item in enumerate(ds['train']):
    split_name = "train" if idx in train_set else "val"
    img = item[image_col]
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(f"dataset/images/{split_name}/img_{idx}.jpg")

print(f"Train: {len(train_set)} | Val: {len(all_idx) - len(train_set)} images saved.")

## Step 4: Auto-Annotate with Grounding DINO

> ⏳ Slowest step — ~10-20 min on T4 GPU.

In [ ]:
import glob
from transformers import pipeline

device = "cuda"
print("Loading Grounding DINO...")
dino = pipeline(
    task="zero-shot-object-detection",
    model="IDEA-Research/grounding-dino-tiny",
    device=device
)

LABELS = ["reflector jacket", "safety vest", "high visibility jacket", "orange safety jacket"]
CONF = 0.25

def annotate_split(split_name):
    paths = sorted(glob.glob(f"dataset/images/{split_name}/*.jpg"))
    annotated = 0
    for i, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        dets = dino(img, candidate_labels=LABELS)
        label_path = f"dataset/labels/{split_name}/{os.path.basename(path).replace('.jpg', '.txt')}"
        lines = []
        for d in dets:
            if d["score"] < CONF:
                continue
            b = d["box"]
            xc = ((b["xmin"] + b["xmax"]) / 2) / w
            yc = ((b["ymin"] + b["ymax"]) / 2) / h
            bw = (b["xmax"] - b["xmin"]) / w
            bh = (b["ymax"] - b["ymin"]) / h
            lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        with open(label_path, "w") as f:
            f.write("\n".join(lines))
        annotated += bool(lines)
        if (i + 1) % 25 == 0:
            print(f"  [{split_name}] {i+1}/{len(paths)}...")
    print(f"  [{split_name}] Done — {annotated}/{len(paths)} images annotated.")

annotate_split("train")
annotate_split("val")
print("Annotation complete!")

## Step 5: Train YOLOv8

In [ ]:
import yaml
from ultralytics import YOLO

dataset_yaml = {
    "path": os.path.abspath("dataset"),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "safety_jacket"}
}
with open("dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

model = YOLO("yolov8n.pt")
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,       # T4 can handle larger batches
    device="cuda",
    patience=20,
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/detect",
    name="safety_jacket_v1"
)
print("Training complete!")

## Step 6: Evaluate & Download `best.pt`

After downloading, place it in your local project at `models/best_jacket.pt`.

In [ ]:
from google.colab import files

best_model_path = "runs/detect/safety_jacket_v1/weights/best.pt"

if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(data="dataset.yaml")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print("\nDownloading best.pt...")
    files.download(best_model_path)
else:
    print(f"Not found: {best_model_path}. Check the runs/ directory.")